In [11]:
import itertools

def evaluate_sentence(sentence, model):
    """
    Evaluates a propositional logic expression under a given truth assignment (model).
    Supported operators: 'and', 'or', 'not', 'implies', 'iff'
    """
    if isinstance(sentence, str):
        return model[sentence]

    op = sentence[0]

    if op == 'not':
        return not evaluate_sentence(sentence[1], model)
    elif op == 'and':
        return all(evaluate_sentence(sub, model) for sub in sentence[1:])
    elif op == 'or':
        return any(evaluate_sentence(sub, model) for sub in sentence[1:])
    elif op == 'implies':
        left = evaluate_sentence(sentence[1], model)
        right = evaluate_sentence(sentence[2], model)
        return (not left) or right
    else:
        raise ValueError(f"Unknown operator: {op}")

def print_truth_table_from_image():
    # Symbols involved
    symbols = ['A', 'B', 'C']

    # Sub-expressions for individual columns
    expr_1 = ('or', 'A', 'C')          # A V C
    expr_2 = ('or', 'B', ('not', 'C')) # B V ~C

    # The Knowledge Base (KB) combines both sentences with an 'and'
    kb_expr = ('and', expr_1, expr_2)

    # The Query (alpha)
    alpha_expr = ('or', 'A', 'B')      # A V B

    # Define widths matching column sizes for perfect alignment
    col_widths = [10, 10, 10, 12, 12, 12, 12]

    # Print Table Header
    headers = ["A", "B", "C", "A V C", "B V ~C", "KB", "alpha"]
    header_line = " | ".join(f"{h:^{w}}" for h, w in zip(headers, col_widths))
    print(header_line)
    print("-" * len(header_line))

    entails = True

    # Generate combinations matching the order in the image (false first, then true)
    # Reversing [False, True] to get false rows printed first
    for combination in itertools.product([False, True], repeat=3):
        model = dict(zip(symbols, combination))

        # Evaluate each required column
        val_A = "true" if model['A'] else "false"
        val_B = "true" if model['B'] else "false"
        val_C = "true" if model['C'] else "false"

        val_expr1 = "true" if evaluate_sentence(expr_1, model) else "false"
        val_expr2 = "true" if evaluate_sentence(expr_2, model) else "false"
        val_kb    = "true" if evaluate_sentence(kb_expr, model) else "false"
        val_alpha = "true" if evaluate_sentence(alpha_expr, model) else "false"

        # Check entailment condition: if KB is true, alpha must be true
        if val_kb == "true" and val_alpha == "false":
            entails = False

        # Print the row values
        row_vals = [val_A, val_B, val_C, val_expr1, val_expr2, val_kb, val_alpha]
        row_line = " | ".join(f"{v:^{w}}" for v, w in zip(row_vals, col_widths))

        # Highlight matching critical lines where KB is true
        if val_kb == "true":
            print(f"{row_line}  <-- (KB is true)")
        else:
            print(row_line)

    print("-" * len(header_line))
    print(f"Checking that KB ⊨ α : {entails}")

# Run the simulation
print_truth_table_from_image()


    A      |     B      |     C      |    A V C     |    B V ~C    |      KB      |    alpha    
------------------------------------------------------------------------------------------------
  false    |   false    |   false    |    false     |     true     |    false     |    false    
  false    |   false    |    true    |     true     |    false     |    false     |    false    
  false    |    true    |   false    |    false     |     true     |    false     |     true    
  false    |    true    |    true    |     true     |     true     |     true     |     true      <-- (KB is true)
   true    |   false    |   false    |     true     |     true     |     true     |     true      <-- (KB is true)
   true    |   false    |    true    |     true     |    false     |    false     |     true    
   true    |    true    |   false    |     true     |     true     |     true     |     true      <-- (KB is true)
   true    |    true    |    true    |     true     |     true     |     